## Enriching stock market data using Open AI API 

<p align="center">
    <img src="images/nasdaq100.png" width="450">
</p>

The Nasdaq-100 is a stock market index made up of 101 equity securities issued by 100 of the largest non-financial companies listed on the Nasdaq stock exchange. It helps investors compare stock prices with previous prices to determine market performance.

In this project you are provided with two CSV files containing Nasdaq-100 stock information:
- _**nasdaq100_CA.csv**_: contains information about companies in the index such as symbol, name, etc. For this analysis, only companies headquartered in California have been selected.
- _**nasdaq100_price_change.csv**_: contains price changes per stock across periods including (but not limited to) one day, five days, one month, six months, one year, etc.

As an AI developer, you will leverage the OpenAI API to classify companies into sectors and produce a summary of sector and company performance for this year, for the companies in the index that are headquartered in California.

# CSV with Nasdaq-100 stock data

In this project, you have available two CSV files `nasdaq100_CA.csv` and `nasdaq100_price_change.csv`.

## nasdaq100_CA.csv

```py
symbol,name,headQuarter,dateFirstAdded,cik,founded
AAPL,Apple Inc.,"Cupertino, CA",,0000320193,1976-04-01
ABNB,Airbnb,"San Francisco, CA",,0001559720,2008-08-01
ADBE,Adobe Inc.,"San Jose, CA",,0000796343,1982-12-01
...
```

## nasdaq100_price_change.csv

```py
symbol,1D,5D,1M,3M,6M,ytd,1Y,3Y,5Y,10Y,max
AAPL,-1.7254,-8.30086,-6.20411,3.042,15.64824,42.99992,8.47941,60.96299,245.42031,976.99441,139245.53954
ABNB,2.1617,-2.21919,9.88336,19.43286,19.64241,68.66902,23.64013,-1.04347,-1.04347,-1.04347,-1.04347
ADBE,0.5409,-1.77817,9.16191,52.0465,38.01522,57.22723,21.96206,17.83037,109.05718,1024.69214,251030.66399
ADI,0.9291,-4.03352,2.58486,3.65887,5.01602,17.02062,8.09735,63.42847,92.81874,286.77518,26012.63736
...
```

In [8]:
# Start your code here!
import os
import pandas as pd
from openai import OpenAI

# Instantiate an API client
client = OpenAI()

# Continue coding here
# Use as many cells as you like

---
## Step 1 — Load the CSV files and add the `ytd` column

**Plan**
- Read `nasdaq100_CA.csv` → `nasdaq100_ca`
- Read `nasdaq100_price_change.csv` → `price_change`
- Merge **only** the `ytd` column into `nasdaq100_ca` on the shared `symbol` key

> We subset `price_change` to `["symbol", "ytd"]` before merging so we don't pull in all the other time-horizon columns.

In [9]:
# Read both CSVs
nasdaq100_ca  = pd.read_csv("nasdaq100_CA.csv")
price_change  = pd.read_csv("nasdaq100_price_change.csv")

# Merge only the ytd column into the main DataFrame
nasdaq100_ca = nasdaq100_ca.merge(
    price_change[["symbol", "ytd"]],
    on="symbol",
    how="left"          # keep every CA company even if missing from price file
)

print(f"Shape: {nasdaq100_ca.shape}")
nasdaq100_ca.head()

Shape: (38, 7)


,symbol,name,headQuarter,dateFirstAdded,cik,founded,ytd
0,AAPL,Apple Inc.,"Cupertino, CA",NaN,320193,1976-04-01,42.99992
1,ABNB,Airbnb,"San Francisco, CA",NaN,1559720,2008-08-01,68.66902
2,ADBE,Adobe Inc.,"San Jose, CA",NaN,796343,1982-12-01,57.22723
3,ADSK,Autodesk,"San Rafael, CA",NaN,769397,1982-01-30,10.02701
4,AMAT,Applied Materials,"Santa Clara, CA",NaN,6951,1967-11-10,55.46366


Quick sanity-check — confirm `ytd` is present and has no unexpected nulls:

In [10]:
print(nasdaq100_ca[["symbol", "ytd"]].isnull().sum())
nasdaq100_ca[["symbol", "name", "ytd"]].sort_values("ytd", ascending=False).head(10)

symbol    0
ytd       0
dtype: int64


,symbol,name,ytd
29,NVDA,Nvidia,217.26511
26,META,Meta Platforms,153.77585
5,AMD,Advanced Micro Devices Inc.,82.45861
25,LRCX,Lam Research,70.25344
1,ABNB,Airbnb,68.66902
8,AVGO,Broadcom Inc.,62.07632
2,ADBE,Adobe Inc.,57.22723
30,PANW,Palo Alto Networks,55.46407
4,AMAT,Applied Materials,55.46366
28,NFLX,Netflix,49.43550


---
## Step 2 — Classify each stock into a sector using OpenAI

**Valid sectors**
```
Technology | Consumer Cyclical | Industrials | Utilities | Healthcare
Communication | Energy | Consumer Defensive | Real Estate | Financial
```

**Approach**
- Build a prompt template with a `{company}` placeholder
- Loop over every ticker, call `gpt-4o-mini` with `temperature=0.0` (deterministic — we want consistent labels)
- Write the returned sector string back into `nasdaq100_ca["sector"]` using `.loc`

In [11]:
# Prompt template — {company} is filled in per iteration
classify_prompt = """Classify company {company} into one of the following sectors.
Answer only with the sector name, no extra text:
Technology, Consumer Cyclical, Industrials, Utilities, Healthcare,
Communication, Energy, Consumer Defensive, Real Estate, or Financial."""

# Initialise the column so we can write into it safely
nasdaq100_ca["sector"] = None

for company in nasdaq100_ca["symbol"]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": classify_prompt.format(company=company)
            }
        ],
        temperature=0.0   # low temp → consistent, focused output
    )
    sector = response.choices[0].message.content.strip()
    nasdaq100_ca.loc[nasdaq100_ca["symbol"] == company, "sector"] = sector

print("Classification complete.")

Classification complete.


Check the distribution of sectors assigned:

In [12]:
print(nasdaq100_ca["sector"].value_counts())
nasdaq100_ca[["symbol", "name", "ytd", "sector"]].head(10)

Technology           23
Consumer Cyclical     8
Healthcare            6
Energy                1
Name: sector, dtype: int64


,symbol,name,ytd,sector
0,AAPL,Apple Inc.,42.99992,Technology
1,ABNB,Airbnb,68.66902,Consumer Cyclical
2,ADBE,Adobe Inc.,57.22723,Technology
3,ADSK,Autodesk,10.02701,Technology
4,AMAT,Applied Materials,55.46366,Technology
5,AMD,Advanced Micro Devices Inc.,82.45861,Technology
6,AMGN,Amgen,-3.32887,Healthcare
7,ATVI,Activision Blizzard,19.10770,Consumer Cyclical
8,AVGO,Broadcom Inc.,62.07632,Technology
9,CDNS,Cadence Design Systems,45.60261,Technology


---
## Step 3 — Get sector & stock recommendations from OpenAI

We embed the full enriched DataFrame (as a clean text table) into the prompt so the model can reason over actual YTD figures — not just its training knowledge.

The response is stored as `stock_recommendations`.

In [13]:
# Build the prompt — embed the DataFrame as a readable text table
recommend_prompt = f"""Provide summary information about Nasdaq-100 stock performance
year to date (YTD) of companies headquartered in California.
Recommend the two best sectors and two or more companies per sector based on YTD performance.

Company data:
{nasdaq100_ca[["symbol", "name", "ytd", "sector"]].to_string(index=False)}
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": recommend_prompt
        }
    ],
    temperature=0.0
)

stock_recommendations = response.choices[0].message.content
print(stock_recommendations)

### Summary of Nasdaq-100 Stock Performance YTD for California Headquartered Companies

As of the current year, the performance of Nasdaq-100 companies headquartered in California has shown significant variation across different sectors. The standout sectors based on year-to-date (YTD) performance are **Technology** and **Consumer Cyclical**.

#### Top Performing Sectors

1. **Technology**
   - The Technology sector has demonstrated remarkable growth, with several companies achieving substantial YTD gains. This sector is characterized by innovation and strong demand for tech products and services.
   
   **Top Companies in Technology:**
   - **Nvidia (NVDA)**: 217.27%
   - **Meta Platforms (META)**: 153.78%
   - **Advanced Micro Devices (AMD)**: 82.46%
   - **Lam Research (LRCX)**: 70.25%
   - **Applied Materials (AMAT)**: 55.46%
   - **Palo Alto Networks (PANW)**: 55.46%
   - **Adobe Inc. (ADBE)**: 57.23%
   - **Broadcom Inc. (AVGO)**: 62.08%
   - **Apple Inc. (AAPL)**: 43.00%

2. **C

### Summary of Nasdaq-100 Stock Performance YTD for California Headquartered Companies

As of the current year, the performance of Nasdaq-100 companies headquartered in California has shown significant variation across different sectors. The technology sector has dominated the performance metrics, with several companies achieving substantial year-to-date (YTD) gains. 

### Top Performing Sectors

1. **Technology**
   - The technology sector has been the standout performer, with several companies achieving remarkable YTD returns. This sector is characterized by innovation and strong demand for tech products and services.
   
   **Top Companies in Technology:**
   - **Nvidia (NVDA)**: 217.27%
   - **Meta Platforms (META)**: 153.78%
   - **Advanced Micro Devices (AMD)**: 82.46%
   - **Lam Research (LRCX)**: 70.25%
   - **Applied Materials (AMAT)**: 55.46%
   - **Palo Alto Networks (PANW)**: 55.46%
   - **Adobe Inc. (ADBE)**: 57.23%
   - **Broadcom Inc. (AVGO)**: 62.08%
   - **Apple Inc. (AAPL)**: 43.00%
   - **Alphabet Inc. (GOOG & GOOGL)**: 47.09% and 47.59% respectively

2. **Consumer Cyclical**
   - The consumer cyclical sector has also performed well, benefiting from increased consumer spending and a rebound in travel and leisure activities.
   
   **Top Companies in Consumer Cyclical:**
   - **Airbnb (ABNB)**: 68.67%
   - **Netflix (NFLX)**: 49.44%
   - **Electronic Arts (EA)**: 1.02%
   - **Intuit (INTU)**: 29.16%
   - **Monster Beverage (MNST)**: 15.46%
   - **Activision Blizzard (ATVI)**: 19.11%

### Conclusion

Based on the year-to-date performance, the **Technology** and **Consumer Cyclical** sectors are the best-performing sectors among California-headquartered companies in the Nasdaq-100. The technology sector, led by companies like Nvidia and Meta Platforms, has shown exceptional growth, while the consumer cyclical sector has also benefited from a resurgence in consumer activity. Investors may consider focusing on these sectors for potential opportunities.